# Phase 4 (comparison): phase4_augmentation_policy

Pilots the four augmentation policies with the `cnn` (seed 0) and both frozen banks:

| letter | policy |
|---|---|
| A | each minority class up to the training N count |
| B | each minority class up to a fixed fraction of N |
| C | a fixed multiple of each minority class's real count |
| D | as B, keeping the bank beats closest to real ones (DTW screen) |

The phase 4a gate picks the policy; its approval writes the full grid file.

Before running (details in `kaggle/README.md`):

1. Settings: **Internet on**. Accelerator: **GPU**.
2. Add-ons > Secrets: `GITHUB_TOKEN` (Contents: read and write on the repo), ticked for this notebook.
3. Add Input: the team's `mitdb-raw` dataset and the banks dataset (the zip of `data/synthetic/` from the
   final phase 3 runs, uploaded once as a private Kaggle dataset). Set `BANKS_DATASET_DIR` to its folder.
4. Set the parameters in the next cell (including `GIT_NAME` and `GIT_EMAIL`), then Run All.
5. When it finishes, the results are already on GitHub: in your clone, `git pull`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
VARIANTS = ["A"]                      # your letter, or several: ["A", "B", "C", "D"]
PREVIEW = False                       # True = run before the gates; results are marked provisional
REPRESENTATION = ""                   # preview only: the phase1_input_representation letter ("" = the phase 1 gate report's winner)
PREVIEW_BANKS = {}                    # preview only: e.g. {"cvae": "phase3_cvae-A", "wgan_gp": "phase3_wgan_gp-B"}
BANKS_DATASET_DIR = ""                # the attached Kaggle dataset with the frozen banks (zip of data/synthetic/), e.g. "/kaggle/input/ecgaug-banks"
SMOKE_TEST = False                    # True = quick check on fake data (a minute or two, no results)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"
PUSH_RESULTS = False                  # True = commit and push the results to GIT_REF from Kaggle; default: ingest the zip yourself
GIT_NAME = ""                         # your name for the results commit, e.g. "Akash Saha"
GIT_EMAIL = ""                        # your GitHub email for the results commit


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
# Turn the variant letters into config files, e.g. "A" -> configs/phase4_augmentation_policy/A_....yaml
configs = []
for letter in VARIANTS:
    (path,) = sorted((REPO / "configs/phase4_augmentation_policy").glob(f"{letter}_*.yaml"))
    configs.append(str(path.relative_to(REPO)))
print("configs:", configs)

extra = ["--device", DEVICE]
if PREVIEW:
    extra.append("--preview")
    if REPRESENTATION:
        extra += ["--representation", REPRESENTATION]
    extra += [arg for family, gid in PREVIEW_BANKS.items() for arg in ("--bank", f"{family}={gid}")]
if SMOKE_TEST:
    extra.append("--smoke")
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase4_augmentation_policy", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs]
cmd += ["--raw-dir", RAW_DATASET_DIR]   # the MIT-BIH files are copied from here and checked against the committed checksums
if BANKS_DATASET_DIR:
    cmd += ["--artifacts-dir", BANKS_DATASET_DIR]
if PUSH_RESULTS and not SMOKE_TEST:
    branch = GIT_REF if len(GIT_REF) != 40 else "kaggle-results"
    cmd += ["--push-to", branch, "--git-name", GIT_NAME, "--git-email", GIT_EMAIL]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode   # env carries the token for the push
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
